In [1]:
import re
import warnings
import pyreadr
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler
import statsmodels.tools.sm_exceptions as sm_exceptions
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [2]:
# loading phenotype data
phenoPath = "C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/00databases/00phenotype/00vabb/01PhenoMeta_assoc_08032025.csv"
phenoData = pd.read_csv(phenoPath)
# Create suffixes
suffixes = ["_9", "_24", "_25", "_11"]
# Replicate and modify pheno_data
phenoData = pd.concat([
    phenoData.assign(SampleID="Sample" + phenoData["SampleID"].astype(str) + suffix)
    for suffix in suffixes
], ignore_index=True)

# Load data
dpclo = pd.read_csv("C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/90summaries/00deepclock-08032025.csv")
# Rename and keep only SampleID and KPANN_brain columns
dpclo = dpclo.rename(columns={'ID': 'SampleID', 'Predicted': 'KPANN_brain_clock'})[['SampleID', 'KPANN_brain_clock']]
# Load cell types and SVAs
cells = pd.read_csv("C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/05celltype-08042025/00vabb_cellprop_svas-08042025.csv")
cells = cells.rename(columns={'name': 'SampleID'})
# Load RNAgecalc
rnaage = pd.read_csv("C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/02trans_age-07272025/01vabb_rnaagecalc-07272025.csv")
rnaage = rnaage.rename(columns={'sample': 'SampleID'})
# Replace the letter between numbers with an underscore
rnaage["SampleID"] = rnaage["SampleID"].str.replace(r"(?<=\d)[A-Z]+(?=\d)", "_", regex=True)
rnaage.columns = [
    col if col == "SampleID" else f"{col}_clock"
    for col in rnaage.columns
]

# Merge in sequence after conversion
merged_df_com = pd.merge(phenoData, dpclo, on='SampleID')
merged_df_com = pd.merge(merged_df_com, cells, on='SampleID')
merged_df_com = pd.merge(merged_df_com, rnaage, on='SampleID')
data_assoc = merged_df_com
data_assoc = data_assoc.dropna()
data_assoc.head()

,SampleID,Cohort,Sex,PMI,RIN,AgeDeath,MOD,COD,COD_drugs_bi,COD_Suicide_bi,...,Peters_uterus_clock,all_uterus_clock,DESeq2_vagina_clock,Pearson_vagina_clock,Dev_vagina_clock,deMagalhaes_vagina_clock,GenAge_vagina_clock,GTExAge_vagina_clock,Peters_vagina_clock,all_vagina_clock
0,Sample5205_9,Lieber,Male,29.0,8.4,51.23,Natural,HypertensiveAtheroscleroticCardiovascularDisease,No,No,...,-59.097454,43.963876,51.171687,36.596403,38.567926,57.057956,25.227871,127.956135,47.407808,22.024076
1,Sample5228_9,Lieber,Female,24.5,6.3,57.83,Accident,MultipleInjuries,No,No,...,-65.220511,43.315390,47.792059,31.681567,39.486125,57.489420,-10.574932,126.123888,47.680470,15.537507
3,Sample5287_9,Lieber,Male,28.0,7.7,40.08,Natural,DilatedCardiomegaly,No,No,...,-58.382340,50.546308,53.894481,34.481126,41.308928,57.179272,56.182066,133.922059,52.355244,22.530600
4,Sample5323_9,Lieber,Male,21.5,7.0,34.57,Accident,MultipleInjuries,No,No,...,-75.125888,38.672864,42.497593,26.557731,37.217969,59.280642,56.126465,124.505995,45.798002,13.986897
5,Sample5368_9,Lieber,Male,28.0,8.2,52.92,Accident,MultipleInjuries,No,No,...,-70.651757,32.901363,28.052320,13.132154,33.925527,56.509115,6.782546,108.710023,41.742782,5.705246


In [3]:
# Standarizing all covariates
# Select only numeric columns
numeric_cols = data_assoc.select_dtypes(include='number').columns
# Scale
scaler = StandardScaler()
data_assoc = data_assoc.copy()
data_assoc.loc[:, numeric_cols] = scaler.fit_transform(data_assoc[numeric_cols])

In [ ]:
# Copy your data
binary_df = data_assoc.copy()

# Define clocks and phenotype columns
clock_cols = [col for col in binary_df.columns if col.endswith('_clock') and pd.api.types.is_numeric_dtype(binary_df[col])]
phenotype_cols = [
    "COD_drugs_bi",
    "COD_Suicide_bi",
    "PsychiatriDisor",
    "MDD",
    "PTSD",
    "LifetimeAntipsych",
    "LifetimeAnticonvulsant",
    "LifetimeAntidepress",
    "LifetimeLithium",
    "Tobbaco_ATOD",
    "Alcohol",
    "Opioid",
    "Opioid_Tox_bi",
    "Sedative_hypnotics",
    "Cannabis",
    "Amphetamine",
    "Cocaine",
    "Hallucinogens",
    "Inhalant",
    "Polysubstance"
]

# Convert Gender (and any other categorical covariates) to numeric BEFORE correlation
covariates = ['AgeDeath', 'Sex', 'RIN', 'PMI', 'ast', 'end', 'mic', 'neu', 'oli', 'opc',' W_1', 'W_2']

# Columns to convert
cols_to_convert = phenotype_cols + ['Sex']

for col in cols_to_convert:
    if col in binary_df.columns:
        # If bool, convert directly to int
        if binary_df[col].dtype == 'bool':
            binary_df[col] = binary_df[col].astype(int)
        # If object/string, convert 'Yes'/'No' and also 'Male'/'Female' for Gender
        elif binary_df[col].dtype == 'object':
            if col == 'Sex':
                binary_df[col] = binary_df[col].map({'Male': 1, 'Female': 0})
            else:
                binary_df[col] = binary_df[col].map({'Yes': 1, 'No': 0})
        else:
            # If already numeric, no change
            pass

# Quick check of unique values after conversion
for col in cols_to_convert:
    if col in binary_df.columns:
        print(f"{col} unique values: {binary_df[col].unique()}")

## Running models
# Adding a dataframe to store the results
results = []
# Loooping in all phenotypes
for pheno in phenotype_cols:
    for clock in clock_cols:
        try:
            formula = f"{pheno} ~ {clock} + {' + '.join(covariates)}"
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")  # suppress all warnings
                
                # Also suppress specific statsmodels warnings if you want
                warnings.filterwarnings("ignore", category=sm_exceptions.ConvergenceWarning)
                warnings.filterwarnings("ignore", category=RuntimeWarning)
                
                model = smf.logit(formula=formula, data=binary_df).fit(disp=0)
            
            # Check convergence
            converged = model.mle_retvals.get('converged', True)
            
            if not converged:
                results.append({
                    'Phenotype': pheno,
                    'Clock': clock,
                    'Beta': np.nan,
                    'SE': np.nan,
                    'z-value': np.nan,
                    'P-value': np.nan,
                    'CI_lower_95': np.nan,
                    'CI_upper_95': np.nan,
                    'Converged': False
                })
                print(f"Model did NOT converge for {pheno} ~ {clock}")
            else:
                coef = model.params[clock]
                pval = model.pvalues[clock]
                conf_int = model.conf_int().loc[clock]
                zval = model.tvalues[clock]
                
                results.append({
                    'Phenotype': pheno,
                    'Clock': clock,
                    'Beta': coef,
                    'SE': model.bse[clock],
                    'z-value': zval,
                    'P-value': pval,
                    'CI_lower_95': conf_int[0],
                    'CI_upper_95': conf_int[1],
                    'Converged': True
                })
        except Exception as e:
            print(f"Model failed for {pheno} ~ {clock}: {e}")
            results.append({
                'Phenotype': pheno,
                'Clock': clock,
                'Beta': np.nan,
                'SE': np.nan,
                'z-value': np.nan,
                'P-value': np.nan,
                'CI_lower_95': np.nan,
                'CI_upper_95': np.nan,
                'Converged': False
            })

results_df = pd.DataFrame(results)

In [35]:
### Storing results
# Extract tissue name from the clock column
def extract_tissue(clock_name):
    # Match the pattern: anything between two underscores and before "_clock"
    match = re.search(r'_(.*?)_clock$', clock_name)
    if match:
        return match.group(1)
    else:
        return None  # fallback in case the format doesn't match

# Add 'Tissue' column to results DataFrame
results_df['Tissue'] = results_df['Clock'].apply(extract_tissue)

# Save results with tissue information
results_df.to_csv("C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/90summaries/03vabb_assoc-08052025.csv", index=False)
results_df

,Phenotype,Clock,Beta,SE,z-value,P-value,CI_lower_95,CI_upper_95,Converged,Tissue
0,COD_drugs_bi,KPANN_brain_clock,-0.243962,0.149831,-1.628247,0.103473,-0.537625,0.049701,True,brain
1,COD_drugs_bi,DESeq2_adipose_tissue_clock,-0.055117,0.142021,-0.388093,0.697947,-0.333472,0.223238,True,adipose_tissue
2,COD_drugs_bi,Pearson_adipose_tissue_clock,0.057320,0.146028,0.392527,0.694669,-0.228889,0.343528,True,adipose_tissue
3,COD_drugs_bi,Dev_adipose_tissue_clock,0.139715,0.176112,0.793330,0.427585,-0.205458,0.484889,True,adipose_tissue
4,COD_drugs_bi,deMagalhaes_adipose_tissue_clock,-0.332512,0.193195,-1.721123,0.085229,-0.711167,0.046143,True,adipose_tissue
...,...,...,...,...,...,...,...,...,...,...
4175,Polysubstance,deMagalhaes_vagina_clock,-0.190829,0.166583,-1.145548,0.251982,-0.517326,0.135668,True,vagina
4176,Polysubstance,GenAge_vagina_clock,-0.160499,0.147900,-1.085185,0.277840,-0.450379,0.129380,True,vagina
4177,Polysubstance,GTExAge_vagina_clock,-0.070260,0.171736,-0.409118,0.682453,-0.406857,0.266337,True,vagina
4178,Polysubstance,Peters_vagina_clock,0.035440,0.143534,0.246909,0.804978,-0.245881,0.316761,True,vagina


In [4]:
# Step 0: Copy and prepare the data
binary_df = data_assoc.copy()

# Step 1: Extract clock and phenotype columns
clock_cols = [col for col in binary_df.columns if col.endswith('_clock') and pd.api.types.is_numeric_dtype(binary_df[col])]
phenotype_cols = [
    "COD_drugs_bi", "COD_Suicide_bi", "PsychiatriDisor", "MDD", "PTSD",
    "LifetimeAntipsych", "LifetimeAnticonvulsant", "LifetimeAntidepress", "LifetimeLithium",
    "Tobbaco_ATOD", "Alcohol", "Opioid", "Opioid_Tox_bi", "Sedative_hypnotics",
    "Cannabis", "Amphetamine", "Cocaine", "Hallucinogens", "Inhalant", "Polysubstance"
]
covariates = ['AgeDeath', 'Sex', 'RIN', 'PMI', 'ast', 'end', 'mic', 'neu', 'oli', 'opc', ' W_1', 'W_2']
cols_to_convert = phenotype_cols + ['Sex']

# Step 2: Convert categorical columns
for col in cols_to_convert:
    if col in binary_df.columns:
        if binary_df[col].dtype == 'bool':
            binary_df[col] = binary_df[col].astype(int)
        elif binary_df[col].dtype == 'object':
            if col == 'Sex':
                binary_df[col] = binary_df[col].map({'Male': 1, 'Female': 0})
            else:
                binary_df[col] = binary_df[col].map({'Yes': 1, 'No': 0})

# Step 3: Create region from SampleID
def extract_region(s):
    if isinstance(s, str):
        if s.endswith('_9'):
            return 'BA9'
        elif s.endswith('_25'):
            return 'BA25'
        elif s.endswith('_11'):
            return 'BA11'
        elif s.endswith('_24'):
            return 'BA24'
        # Add more as needed
    return 'Unknown'

binary_df['Region'] = binary_df['SampleID'].apply(extract_region)

# Step 4: Run stratified logistic regression
results2 = []

for region, group_df in binary_df.groupby('Region'):
    print(f"\nRunning models for region: {region}")
    
    for pheno in phenotype_cols:
        for clock in clock_cols:
            try:
                formula = f"{pheno} ~ {clock} + {' + '.join(covariates)}"
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    warnings.filterwarnings("ignore", category=sm_exceptions.ConvergenceWarning)
                    warnings.filterwarnings("ignore", category=RuntimeWarning)

                    model = smf.logit(formula=formula, data=group_df).fit(disp=0)

                converged = model.mle_retvals.get('converged', True)

                if not converged:
                    print(f"Model did NOT converge for {pheno} ~ {clock} in {region}")
                    results2.append({
                        'Region': region, 'Phenotype': pheno, 'Clock': clock,
                        'Beta': np.nan, 'SE': np.nan, 'z-value': np.nan,
                        'P-value': np.nan, 'CI_lower_95': np.nan, 'CI_upper_95': np.nan,
                        'Converged': False
                    })
                else:
                    coef = model.params[clock]
                    pval = model.pvalues[clock]
                    conf_int = model.conf_int().loc[clock]
                    zval = model.tvalues[clock]

                    results2.append({
                        'Region': region, 'Phenotype': pheno, 'Clock': clock,
                        'Beta': coef, 'SE': model.bse[clock], 'z-value': zval,
                        'P-value': pval, 'CI_lower_95': conf_int[0],
                        'CI_upper_95': conf_int[1], 'Converged': True
                    })
            except Exception as e:
                print(f"Model failed for {pheno} ~ {clock} in {region}: {e}")
                results2.append({
                    'Region': region, 'Phenotype': pheno, 'Clock': clock,
                    'Beta': np.nan, 'SE': np.nan, 'z-value': np.nan,
                    'P-value': np.nan, 'CI_lower_95': np.nan, 'CI_upper_95': np.nan,
                    'Converged': False
                })

# Step 5: Convert results to DataFrame
results2_df = pd.DataFrame(results2)
results2_df


Running models for region: BA11
Model failed for COD_Suicide_bi ~ KPANN_brain_clock in BA11: Singular matrix
Model did NOT converge for COD_Suicide_bi ~ DESeq2_adipose_tissue_clock in BA11
Model failed for COD_Suicide_bi ~ Pearson_adipose_tissue_clock in BA11: Singular matrix
Model failed for COD_Suicide_bi ~ Dev_adipose_tissue_clock in BA11: Singular matrix
Model failed for COD_Suicide_bi ~ deMagalhaes_adipose_tissue_clock in BA11: Singular matrix
Model failed for COD_Suicide_bi ~ GenAge_adipose_tissue_clock in BA11: Singular matrix
Model failed for COD_Suicide_bi ~ GTExAge_adipose_tissue_clock in BA11: Singular matrix
Model failed for COD_Suicide_bi ~ Peters_adipose_tissue_clock in BA11: Singular matrix
Model failed for COD_Suicide_bi ~ all_adipose_tissue_clock in BA11: Singular matrix
Model failed for COD_Suicide_bi ~ DESeq2_adrenal_gland_clock in BA11: Singular matrix
Model failed for COD_Suicide_bi ~ Pearson_adrenal_gland_clock in BA11: Singular matrix
Model failed for COD_Suicid

,Region,Phenotype,Clock,Beta,SE,z-value,P-value,CI_lower_95,CI_upper_95,Converged
0,BA11,COD_drugs_bi,KPANN_brain_clock,-0.712371,0.499100,-1.427309,0.153491,-1.690590,0.265848,True
1,BA11,COD_drugs_bi,DESeq2_adipose_tissue_clock,0.110510,0.460793,0.239827,0.810465,-0.792628,1.013649,True
2,BA11,COD_drugs_bi,Pearson_adipose_tissue_clock,0.472380,0.487537,0.968912,0.332589,-0.483174,1.427935,True
3,BA11,COD_drugs_bi,Dev_adipose_tissue_clock,0.684255,0.575264,1.189462,0.234258,-0.443242,1.811753,True
4,BA11,COD_drugs_bi,deMagalhaes_adipose_tissue_clock,-1.107714,0.753246,-1.470586,0.141403,-2.584049,0.368622,True
...,...,...,...,...,...,...,...,...,...,...
16715,BA9,Polysubstance,deMagalhaes_vagina_clock,-0.330854,0.417579,-0.792315,0.428177,-1.149293,0.487586,True
16716,BA9,Polysubstance,GenAge_vagina_clock,-0.054901,0.260024,-0.211138,0.832780,-0.564540,0.454738,True
16717,BA9,Polysubstance,GTExAge_vagina_clock,-0.055596,0.505343,-0.110016,0.912397,-1.046050,0.934858,True
16718,BA9,Polysubstance,Peters_vagina_clock,0.396472,0.457815,0.866008,0.386486,-0.500830,1.293773,True


In [8]:
### Storing results
# Extract tissue name from the clock column
def extract_tissue(clock_name):
    # Match the pattern: anything between two underscores and before "_clock"
    match = re.search(r'_(.*?)_clock$', clock_name)
    if match:
        return match.group(1)
    else:
        return None  # fallback in case the format doesn't match

# Add 'Tissue' column to results DataFrame
results2_df['Tissue'] = results2_df['Clock'].apply(extract_tissue)

# Save results with tissue information
results2_df.to_csv("C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/90summaries/04vabb_assoc-08062025.csv", index=False)
results2_df

,Region,Phenotype,Clock,Beta,SE,z-value,P-value,CI_lower_95,CI_upper_95,Converged,Tissue
0,BA11,COD_drugs_bi,KPANN_brain_clock,-0.712371,0.499100,-1.427309,0.153491,-1.690590,0.265848,True,brain
1,BA11,COD_drugs_bi,DESeq2_adipose_tissue_clock,0.110510,0.460793,0.239827,0.810465,-0.792628,1.013649,True,adipose_tissue
2,BA11,COD_drugs_bi,Pearson_adipose_tissue_clock,0.472380,0.487537,0.968912,0.332589,-0.483174,1.427935,True,adipose_tissue
3,BA11,COD_drugs_bi,Dev_adipose_tissue_clock,0.684255,0.575264,1.189462,0.234258,-0.443242,1.811753,True,adipose_tissue
4,BA11,COD_drugs_bi,deMagalhaes_adipose_tissue_clock,-1.107714,0.753246,-1.470586,0.141403,-2.584049,0.368622,True,adipose_tissue
...,...,...,...,...,...,...,...,...,...,...,...
16715,BA9,Polysubstance,deMagalhaes_vagina_clock,-0.330854,0.417579,-0.792315,0.428177,-1.149293,0.487586,True,vagina
16716,BA9,Polysubstance,GenAge_vagina_clock,-0.054901,0.260024,-0.211138,0.832780,-0.564540,0.454738,True,vagina
16717,BA9,Polysubstance,GTExAge_vagina_clock,-0.055596,0.505343,-0.110016,0.912397,-1.046050,0.934858,True,vagina
16718,BA9,Polysubstance,Peters_vagina_clock,0.396472,0.457815,0.866008,0.386486,-0.500830,1.293773,True,vagina
